# 第11回　教師なし学習
***
> **前提**: 第6回までの教師あり学習に続き，ラベルなしデータの分析を学びます。

## 目次
1. k-means クラスタリング
2. クラスタ数の評価
3. PCA による次元削減
4. 2次元可視化

---

## この回で学ぶこと

### 教師あり学習 vs 教師なし学習

これまでの学習（第1〜10回）は，すべて「正解ラベル」がある**教師あり学習**だった。

```
教師あり学習: データ + 正解ラベル → モデルが「入力→出力」の関係を学習
教師なし学習: データのみ           → データ内の隠れた構造・パターンを発見
```

現実には「正解ラベルがない」データの方が圧倒的に多い。センサーデータ，SNS投稿，画像，遺伝子発現データなど，ラベル付けコストが高いデータには教師なし学習が不可欠だ。

### k-means クラスタリング

k-means の仕組みを直感的に理解しよう：

1. ランダムにk個の「重心（centroid）」を配置
2. 各データ点を最も近い重心に割り当て（クラスタを形成）
3. 各クラスタの重心を再計算
4. 変化がなくなるまで2〜3を繰り返す

**重要な特性**：
- 初期値依存性：`random_state` を固定しないと毎回結果が異なる
- k は事前に決める必要がある（最大の欠点）
- 球形クラスタ向け：細長い，三日月形などのクラスタは苦手
- スケールに敏感：`StandardScaler` で正規化が必須

### シルエットスコアとは

シルエットスコアは「各データ点が自分のクラスタに適切に属しているか」を -1〜1 の値で評価する指標だ：
- **値が1に近い**: そのデータ点は自分のクラスタの中心に近く，他のクラスタから遠い → 良い分類
- **値が0に近い**: クラスタの境界付近にある
- **値が負**: 本来は別のクラスタに属すべきかもしれない

### 主成分分析（PCA）とは

高次元データ（例：30変数）を低次元（2次元）に圧縮する手法だ。単純に変数を削除するのではなく，**データの分散が最大になる方向（主成分）を見つけ**，その方向に投影する。

活用場面：
- **可視化**: 30次元のデータを2次元にして散布図で確認
- **前処理**: 次元を減らして計算コストを削減，ノイズ除去
- **多重共線性の解決**: 相関する変数をまとめる

> **卒業研究での活用例**: テキストデータの単語ベクトル（数千次元）を PCA で2〜3次元に圧縮し，似た意味の単語をクラスタリングする研究はよく見られる。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris, load_wine
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


## 問題1　k-means クラスタリング
***

### Iris データセットについて

Iris データセットは機械学習の入門で最もよく使われるデータセットだ：
- 150件のアヤメの花のデータ
- 4特徴量：がく片の長さ/幅，花びらの長さ/幅（単位：cm）
- 3クラス：Setosa / Versicolor / Virginica

今回は「正解ラベル（品種）を使わずに」，特徴量だけから3つのグループを自動発見できるかを試す。発見したクラスタが実際の品種と一致すれば，この手法の有効性が確認できる。

### なぜ標準化が必要か

Iris の4特徴量はどれもcm単位で似たスケールだが，k-means はユークリッド距離を使うため，スケールが大きい変数の影響が過大になる。**標準化は k-means の前処理として必須**だと覚えておこう。

### クラスタラベルの注意点

k-means のラベルは 0, 1, 2 が割り当てられるが，この番号と品種の名前は対応していない。例えば「クラスタ0 = Setosa」とは限らず，実行するたびに対応が変わりうる（番号に意味はない）。

### 課題

Iris データセット `load_iris()` の特徴量を `StandardScaler` で標準化し，`KMeans(n_clusters=3, random_state=0)` でクラスタリングしてください。

各サンプルのクラスタラベル（先頭10件）と **シルエットスコア**を出力してください。

> **観察ポイント**: シルエットスコアが0.5以上であれば，比較的良いクラスタリングと言える。Iris データは品種間の境界が明確なので，高いスコアが期待できる。

#### Hints
- `KMeans` は `fit_predict` メソッドで学習とラベル割り当てを同時に行える（`fit` → `labels_` でも同じ結果）
- `silhouette_score` の引数は `(特徴量行列, クラスタラベル)` の順
- シルエットスコアは -1〜1 の範囲で、高いほどクラスタが明確に分離されていることを意味する

In [ ]:
# k-means クラスタリング
# ここにあなたのコードを書いてください


## 問題2　最適なクラスタ数の探索
***

### k をどう決めるか

k-means の最大の課題は「k（クラスタ数）を事前に決めなければならない」点だ。データを見ても適切な k が分からない場合，以下の手法で探索する：

**① エルボー法（Elbow Method）**
- 各 k の「クラスタ内誤差の合計（inertia）」をプロット
- グラフが「肘（elbow）」のように急に折れ曲がる点が最適 k の目安
- `kmeans.inertia_` で取得できる

**② シルエットスコア法**
- 各 k のシルエットスコアをプロット
- **スコアが最も高い k** が最適（今回使用）

> **どちらを使うべき?** 2つの方法が一致する k があれば信頼性が高い。一致しない場合はドメイン知識（そのデータについての専門知識）も参考にする。

### 課題

クラスタ数 k を 2〜6 で変化させ，各 k に対するシルエットスコアを計算し，最もスコアが高い k を出力してください。

横軸: k，縦軸: シルエットスコア の折れ線グラフを描画してください。

> **確認ポイント**: 正解ラベルが3クラスあることを知っている今，k=3 がベストスコアになるか確認しよう。もしなれば k-means が品種の境界を正しく捉えていることを意味する。


In [ ]:
# クラスタ数の評価
# ここにあなたのコードを書いてください


## 問題3　PCA による次元削減
***

### Wine データセットについて

- 3種類のワインの化学分析データ（178件）
- 13特徴量：アルコール度数，リンゴ酸，灰分，マグネシウム，フェノール類，フラボノイドなど
- 多くの特徴量が互いに相関している（例：フェノール類とフラボノイド）

PCA は相関する変数をうまく圧縮するのが得意だ。13次元の情報をどれだけ2次元で保持できるか確認しよう。

### 寄与率（説明分散比）の意味

`pca.explained_variance_ratio_` は各主成分が「元データの全分散のうち何%を説明するか」を示す：
- 第1主成分：データの変動が最も大きい方向
- 第2主成分：第1主成分と直交する方向で，次に変動が大きい方向

```
例: explained_variance_ratio_ = [0.36, 0.19]
→ 第1主成分で 36%, 第2主成分で 19% の情報を保持
→ 合計 55% の情報で 13 次元 → 2 次元に圧縮
```

**累積寄与率**（合計）が70〜80%以上あれば，2次元での可視化に十分な情報が保持されている。

### PCA の前に標準化が必須な理由

PCA は「分散が大きい方向」を主成分として選ぶ。スケールの大きい特徴量（例：マグネシウム：70〜162mg）は自動的に「重要な方向」として選ばれてしまう。標準化することで，すべての特徴量を公平に扱える。

### 課題

Wine データセット `load_wine()` を標準化し，`PCA(n_components=2)` で2次元に次元削減してください。

変換後のデータの形状（shape）と，第1主成分・第2主成分の**寄与率**と**累積寄与率**を出力してください。

#### Hints
- `PCA` も scikit-learn の変換器なので、`fit_transform` で標準化済みデータを変換できる
- 変換後の寄与率は `pca` オブジェクトの属性として格納されている（`explained_variance_ratio_`）
- 各主成分の寄与率を足し合わせると累積寄与率になる。NumPy の集計関数が使える

In [ ]:
# PCA
# ここにあなたのコードを書いてください


## 問題4　PCA 結果の可視化と解釈
***

### 可視化で何を確認するか

PCA 後の2次元散布図を色分けして描くことで：
1. **クラスが分離できているか**：色が綺麗に分かれていれば，PCA が識別に有効な方向を捉えている
2. **外れ値の存在**：散布図の端に孤立した点がないか
3. **クラス間の重なり**：重なりが多い場合，このデータは2次元では分離が難しい

### この可視化の意義

もし3クラスが2次元平面上で綺麗に分離できていれば，わずか2つの主成分（元の13特徴量の線形結合）が品種分類に十分な情報を持っていることを意味する。これは：
- 教師なし学習（今回）：どのクラスタに属するかを探索的に確認
- 次元削減の前処理として：k-means クラスタリングの前に PCA を適用することもある

### 課題

問題3の2次元データを散布図で可視化してください。色分けには Wine データの正解ラベル（`target`）を用いてください。

横軸: 第1主成分（寄与率を軸ラベルに含める），縦軸: 第2主成分，凡例あり。

> **発展**: `seaborn.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=y)` を使うとより見やすい散布図が描ける。また，各主成分がどの元変数と強く対応するかは `pca.components_` で確認できる。


In [ ]:
# PCA 2次元可視化
# ここにあなたのコードを書いてください
